# Configs

In [0]:
# print('PLACEHOLDER for checking Spark configs')

## Use built-in functions

In [0]:
from pyspark.sql.functions import broadcast

## Constants

In [0]:
# STATUS__ORDER__CANCELLED = 'Cancelled'
# STATUS__ORDER_ITEM__CANCELLED = 'Cancelled'

CATALOG_NAME = 'workspace'
SCHEMA_NAME__BRONZE = 'olist_bronze'

# Execute common notebook

In [0]:
notebook_path = '/Workspace/Users/lalitstar@gmail.com/olist_dataset_databricks/raw/dataset_investigation.py'
# dbutils.notebook.run(notebook_path, timeout_seconds=300)

In [0]:
%run /Workspace/Users/lalitstar@gmail.com/olist_dataset_databricks/raw/dataset_investigation.py

# Create/Replace bronze schema


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

qualified_bronze_schema_name = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qualified_bronze_schema_name}")

# Save Master Datasets

### bronze.Status

In [0]:
# Write DataFrame to Unity Catalog in Delta format
table__bronze__status = f"{CATALOG_NAME}.{SCHEMA_NAME}.status"
status_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__bronze__status)


### bronze.Products

In [0]:
table__bronze__products = f"{CATALOG_NAME}.{SCHEMA_NAME}.products"
prod_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__bronze__products)

### bronze.Customers

In [0]:
table__bronze__customers = f"{CATALOG_NAME}.{SCHEMA_NAME}.customers"
cust_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__bronze__customers)

### Confirm master tables in bronze schema

In [0]:
%sql
select type, count(*) total_status from workspace.olist_bronze.status group by type;

In [0]:
%sql
-- select count(*) from workspace.olist_bronze.products;

WITH 
PRODUCT_AVAILABILITY AS (
    SELECT 
        CASE 
            WHEN deleted_at IS NULL THEN 'Active' 
            ELSE 'Inactive' 
        END AS product_availability,
        product_sku
    FROM 
        workspace.olist_bronze.products
)
SELECT 
    COUNT(*) AS product_count,
    product_availability
FROM 
    PRODUCT_AVAILABILITY
GROUP BY
    product_availability;
    


In [0]:
%sql
select count(*) customer_count from workspace.olist_bronze.customers;

# Save Transactional Datasets

## bronze.Orders

In [0]:
table__bronze__orders = f"{CATALOG_NAME}.{SCHEMA_NAME}.orders"
orders_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__bronze__orders)

## bronze.Order_Items

In [0]:
table__bronze__order_items = f"{CATALOG_NAME}.{SCHEMA_NAME}.order_items"
order_items_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__bronze__order_items)

## Confirm transaction tables in bronze schema

In [0]:
%sql
select year, month, count(*) total_orders from workspace.olist_bronze.orders group by year, month order by year, month;
    


In [0]:
%sql
select year, month, count(*) total_items from workspace.olist_bronze.order_items group by year, month order by year